In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Tue Oct 22 20:48:49 2024

@author: yuri

email: syuri@tju.edu.cn
"""





#%% GLOBAL VARIABLE
NEEDTRAIN = True        # train the model
LOAD_MODELS_FROM = 'modelsxxxxxxx'       # the path of trained model weights for testing

import os
os.environ["KERAS_BACKEND"] = "tensorflow"
if os.getcwd().split(os.sep)[1] == 'home':
    PLATFORM = 'local'  # *** local kaggle *** local training or online testing
    for dir_name in os.listdir('./input/'):
        if dir_name[:6] == 'models':
            LOAD_MODELS_FROM = dir_name
elif os.getcwd().split(os.sep)[1] == 'kaggle':
    PLATFORM = 'kaggle' # *** local kaggle *** local training or online testing
    NEEDTRAIN = False
    for dir_name in os.listdir('/kaggle/input/'):
        if dir_name[:6] == 'models':
            LOAD_MODELS_FROM = dir_name


DATATYPE = ['eeg']    # *** spe, eeg, stft, img *** the data type used
print(DATATYPE)


if PLATFORM == 'local':
    LOAD_MODELS_FROM = f'./input/{LOAD_MODELS_FROM}'
    LOAD_DATA_FROM = './input/hms-harmful-brain-activity-classification'
elif PLATFORM == 'kaggle':
    LOAD_MODELS_FROM = f'/kaggle/input/{LOAD_MODELS_FROM}'
    LOAD_DATA_FROM = '/kaggle/input/hms-harmful-brain-activity-classification'



SFREQ = 200             # EEG sampling rate
RSFREQ = 200            # resampled EEG sampling rate

EEG_LENGTH = 50         # the length of EEG data used for each sample
EEG_LENGTH_USED = 50
EEG_CHANNEL_USED = 16 # 16 18

EEG_MULTIPLY = 1

IMG_LENGTH = 20
IMG_HIGH = 324
IMG_WIDE = 324

SPE_HIGH = 40           # the height of the spectrogram 100
SPE_WIDE = 1000            # the width of the spectrogram  10 * 30

STFT_LENGTH = 50 
STFT_TIME = 0.1
# STFT_HIGH = 100           # the height of the STFT (eeg spectrogram)
# STFT_WIDE = round(STFT_LENGTH / STFT_TIME)          # the width of the STFT (eeg spectrogram)  50 * 5
STFT_HIGH = 50
STFT_WIDE = 200


filter_range = [0.5, 45]   # eeg filtering range   [0.5, 45] None
filter_range2 = [0.1, 35]    # eeg filtering range

SEED = 2024             # seed

BATCHSIZE = 16        # batch size

LEARN_RATE = 1e-3 * BATCHSIZE / 16
EPOCHS = 15

SPLITS = 5

READ_EEG_FILES = False      # preprocess eeg
READ_SPE_FILES = False     # preprocess spectrogram

spectrograms = {}       # preprocessed spectrograms for training
eegs = {}               # preprocessed eegs for training
stfts = {}              # preprocessed short-time fourier transform plots for training
imgs = {}

spectrograms_test = {}    
eegs_test = {}             
stfts_test = {}     
imgs_test = {}   

# LL RL LP RP
BRAIN = [
         'Fp1-F7', 'F7-T3', 'T3-T5', 'T5-O1',   # LL
         'Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1',   # LP 
         'Fz-Cz', 'Cz-Pz',
         'Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2',   # RP
         'Fp2-F8', 'F8-T4', 'T4-T6', 'T6-O2',   # RL
        ]

TEST_BATCHSIZE = 128



#%% IMPORT
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES']='0, 1'
import warnings
warnings.filterwarnings('ignore')
import io
from PIL import Image
import pandas as pd, numpy as np
from sklearn.metrics import confusion_matrix


from tensorflow.keras import optimizers
from tensorflow.keras.models import clone_model
from tensorflow.python.framework.ops import reset_default_graph

import matplotlib
# matplotlib.use('Agg')
import matplotlib.pyplot as plt

from scipy import signal
from scipy.ndimage import zoom
import time
import gc
import torchaudio
import torch

np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'


import tensorflow as tf
print(tf.version.VERSION)
print(tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)  # 按需分配显存
    except RuntimeError as e:
        print(e)
        
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# use mixed precision
MIX = True
if MIX:
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
else:
    print('Using full precision')
    
if NEEDTRAIN:
    import itertools



#%% LOAD TRAIN DATAFRAME
df = pd.read_csv(os.path.join(LOAD_DATA_FROM, 'train.csv'))
TARGETS = df.columns[-6:]
print('Train shape:', df.shape)
print('Targets', list(TARGETS))






#%% CREATE NON-OVERLAPPING EEG ID TRAIN DATAFRAME
if NEEDTRAIN:
    TARGETS_RAW = list()
    for i in TARGETS:
        TARGETS_RAW.append(i + '_raw')
        
    if READ_EEG_FILES:
        
        # train = pd.DataFrame()
        # for eeg_id in df.eeg_id.unique():
        #     train_temp = df.loc[df.eeg_id == eeg_id]
        #     train_temp_num = round((train_temp.eeg_label_offset_seconds.max() - train_temp.eeg_label_offset_seconds.min() + 50) / 50)
        #     train_temp = train_temp.iloc[np.random.permutation(len(train_temp))].reset_index(drop=True)
        #     train_temp2 = train_temp.loc[:train_temp_num-1]
        #     train = pd.concat((train, train_temp2))
        # train = train.reset_index(drop=True)
            
        train = df.drop_duplicates(['eeg_id', 'seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']).reset_index(drop=True)
        
        train['sign_id'] = train.index.values
        df['sign_id'] = df.index.values
        
        y_data = train[TARGETS].values
        train[TARGETS_RAW] = y_data
        y_data = y_data / y_data.sum(axis=1,keepdims=True)
        train[TARGETS] = y_data

        train.to_csv('train.csv', index=False)
    else:
        train = pd.read_csv('train.csv')






#%% READ TRAIN EEGS
if filter_range != None:
    b, a = signal.butter(3, np.float32(filter_range)*2/RSFREQ, 'bandpass')
    
if NEEDTRAIN:
    PATH = os.path.join(LOAD_DATA_FROM, 'train_eegs') + '/'
    if READ_EEG_FILES:
        if ('stft' in DATATYPE) or ('img' in DATATYPE):
            b2, a2 = signal.butter(3, np.float32(filter_range2)*2/RSFREQ, 'bandpass')
        time_start_time = time.time()
        
        for i, eeg_id in enumerate(train.eeg_id.unique()):
            
            if i%200==0:
                gc.collect()
                xx = (time.time() - time_start_time)
                yy = xx / (i+1) * len(train.eeg_id.unique())
                print(i, f'time: {round(xx / 60, 2)} min / {round(yy / 60, 2)} min')
            eeg_default = pd.read_parquet(os.path.join(PATH, (str(eeg_id) + '.parquet')))
            
            eeg = list()
            for channel in BRAIN:
                eeg_temp = (eeg_default.loc[:, channel.split('-')[0]] - eeg_default.loc[:, channel.split('-')[1]]).values
                eeg_temp[np.isnan(eeg_temp)] = 0
                eeg.append(np.reshape(eeg_temp, (1, -1)))
            eeg = np.concatenate(eeg, axis=0)
            
            if SFREQ != RSFREQ:
                eeg = signal.resample_poly(eeg, RSFREQ, SFREQ, axis=1)
            
            if 'stft' in DATATYPE:
                
                # eeg2 = signal.filtfilt(b, a, eeg, axis=1)

                # nperseg = 128
                # ff, tt, ss = signal.spectrogram(eeg2, axis=1, fs=RSFREQ, nperseg=nperseg,
                #                                 noverlap = round(nperseg - RSFREQ * STFT_TIME),
                #                                 nfft = RSFREQ * 5
                #                                 )
                
                # ss[np.isnan(ss)] = 0
                # ss = ss[:, (ff>0) * (ff<=20), :]
                
                eeg2 = signal.filtfilt(b, a, eeg, axis=1)
                
                eeg2 = torch.from_numpy(eeg2.copy())
                n_fft, win_length, hop_length = 500, 128, 50
                ss = torchaudio.transforms.Spectrogram(n_fft=n_fft, win_length=win_length, hop_length=hop_length, power=1)(eeg2)
                ss = ss.numpy()
                ss = ss[:, :round(20 / (RSFREQ / n_fft)), ]
                
                tt = np.arange(ss.shape[2]) * hop_length / RSFREQ

                ss = np.concatenate((ss[0:round(EEG_CHANNEL_USED/2), :, :], ss[-round(EEG_CHANNEL_USED/2):, :, :]), axis=0)
                
                # for k in range(4):
                #     ss[k, :, :] = np.mean(ss[k*4:(k+1)*4, :, :], 0)

                # ss = ss[:4, :, :]
                
                ss = np.array(ss, dtype=np.float32)
                tt = np.array(tt, dtype=np.float32)


            if 'img' in DATATYPE:
                eeg2 = signal.filtfilt(b2, a2, eeg, axis=1)
            
                eeg2 = np.clip(eeg2, a_min=-1024, a_max=1024)
                
                
                train_plot = train[train.eeg_id == eeg_id].reset_index(drop=True)
                for j in range(len(train_plot)):
                    row = train_plot.iloc[j]
                    rows = df.loc[(df.eeg_id == row.eeg_id) * (df.seizure_vote == row.seizure_vote_raw) * (df.lpd_vote == row.lpd_vote_raw) * (df.gpd_vote == row.gpd_vote_raw) * (df.lrda_vote == row.lrda_vote_raw) * (df.grda_vote == row.grda_vote_raw), :].reset_index(drop=True)
                    row = rows.sort_values(by='eeg_sub_id').reset_index(drop=True).iloc[len(rows)//2]
                    
                    eeg_plot = eeg2[:, round(row.eeg_label_offset_seconds * RSFREQ):round((row.eeg_label_offset_seconds + EEG_LENGTH) * RSFREQ)]
                    eeg_plot = eeg_plot[:, round((EEG_LENGTH - IMG_LENGTH) / 2 * RSFREQ):round((EEG_LENGTH + IMG_LENGTH) / 2 * RSFREQ)]
                    
                    img_save = np.zeros((eeg_plot.shape[0], 36, IMG_WIDE), dtype=np.float32)
                    for ii in range(eeg_plot.shape[0]):
                        fig = plt.figure(clear=True, figsize=(3.93, 2 / 18 * 2))
                        fig.patch.set_facecolor('black')

                        plt.plot(eeg_plot[ii, :] + 100, color='red', linewidth=0.2)
                        
                        plt.xlim(-5, eeg_plot.shape[1] + 5)
                        plt.ylim(0, 200)
                        plt.axis('off')
                        
                        byte_stream = io.BytesIO()
                        plt.savefig(byte_stream, format='png', bbox_inches='tight', dpi=100)
                        byte_stream.seek(0)
                        img = Image.open(byte_stream)
                        img = np.array(img)[:, :, :1]
                        img = img / 255
                        img = np.array(img, dtype=np.float32)
                        byte_stream.truncate()
                        plt.close('all')
                        
                        if img.shape != (36, IMG_WIDE, 1):
                            img = np.concatenate((img, img, img), 2)
                            img = np.array(tf.image.resize(img, (36, IMG_WIDE)), dtype=np.float32)
                        img = img[:, :, 0]
                        
                        img_save[ii, :, :] = img
 
    
                    imgs[train_plot.sign_id[j]] = img_save
            
            eeg = np.clip(eeg, a_min=-1024, a_max=1024)
            if filter_range != None:
                eeg = signal.filtfilt(b, a, eeg, axis=1)
            eeg = np.array(eeg, dtype=np.float32)
            
            if 'eeg' in DATATYPE:
                eegs[eeg_id] = eeg
            if 'stft' in DATATYPE:
                stfts[eeg_id] = ss
                stfts[-eeg_id] = tt
            
                
        
        if not os.path.exists('./input/preprocess'):
            os.makedirs('./input/preprocess')
        if 'eeg' in DATATYPE:
            np.save('./input/preprocess/eegs.npy', eegs, allow_pickle=True)
        if 'stft' in DATATYPE:
            np.save('./input/preprocess/stfts.npy', stfts, allow_pickle=True)
        if 'img' in DATATYPE:
            np.save('./input/preprocess/imgs.npy', imgs, allow_pickle=True)

    else:
        if PLATFORM == 'local':
            datapath = './' + os.path.join('input', 'preprocess')
        elif PLATFORM == 'kaggle':
            datapath = '/kaggle/' + os.path.join('input', 'preprocess')

        if 'eeg' in DATATYPE:
            eegs = np.load(os.path.join(datapath, 'eegs.npy'), allow_pickle=True).item()
        if 'stft' in DATATYPE:
            stfts = np.load(os.path.join(datapath, 'stfts.npy'), allow_pickle=True).item()
        if 'img' in DATATYPE:
            imgs = np.load(os.path.join(datapath, 'imgs.npy'), allow_pickle=True).item()








#%% READ TRAIN SPECTROGRAMS
if NEEDTRAIN:
    PATH = os.path.join(LOAD_DATA_FROM, 'train_spectrograms') + '/'
    files = os.listdir(PATH)
    print(f'There are {len(files)} spectrogram parquets')
    time_start_time = time.time()
    if READ_SPE_FILES:    
        for i, f in enumerate(files):
            if i%200==0:
                gc.collect()
                xx = (time.time() - time_start_time)
                yy = xx / (i+1) * len(files)
                print(i, f'time: {round(xx / 60, 2)} min / {round(yy / 60, 2)} min')
            tmp = pd.read_parquet(f'{PATH}{f}')
            name = int(f.split('.')[0])
            spectrograms[name] = tmp.iloc[:,1:].values
        if not os.path.exists('./input/preprocess'):
            os.makedirs('./input/preprocess')
        np.save('./input/preprocess/spectrograms.npy', spectrograms, allow_pickle=True)
    else:
        if 'spe' in DATATYPE:
            if PLATFORM == 'local':
                spectrograms = np.load('./input/preprocess/spectrograms.npy', allow_pickle=True).item()
            elif PLATFORM == 'kaggle':
                spectrograms = np.load('/kaggle/input/preprocess/spectrograms.npy', allow_pickle=True).item()









#%% DATA GENERATOR
class DataGenerator(tf.keras.utils.Sequence):
    def __init__(self, dataframe, batch_size=32, shuffle=False, sample_weights=False, mode='train',
                 eegs=None, stfts=None, specs=None, imgs=None): 

        self.dataframe = dataframe
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.sample_weights = sample_weights
        self.mode = mode
        self.eegs = eegs
        self.stfts = stfts
        self.specs = specs
        self.imgs = imgs
        self.on_epoch_end()
        
    def __len__(self):
        ct = int( np.ceil( len(self.dataframe) / self.batch_size ) )
        return ct

    def __getitem__(self, index):
        indexes = self.indexes[index*self.batch_size:(index+1)*self.batch_size]
        x, y, sample_weights = self.__data_generation(indexes)
        return x, y, sample_weights

    def on_epoch_end(self):
        self.nan = 0
        self.indexes = np.arange( len(self.dataframe) )
        if self.shuffle: np.random.shuffle(self.indexes)
                        
    def __data_generation(self, indexes):
        if 'spe' in DATATYPE:
            x_spe = np.zeros((len(indexes), 4, SPE_HIGH, SPE_WIDE), dtype='float32')
        if 'eeg' in DATATYPE:
            x_eeg = np.zeros((len(indexes), EEG_CHANNEL_USED * EEG_MULTIPLY, round(EEG_LENGTH_USED * RSFREQ / EEG_MULTIPLY)),dtype='float32')
            # x_eeg10 = np.zeros((len(indexes), EEG_CHANNEL_USED, EEG_CHANNEL_USED),dtype='float32')
        if 'img' in DATATYPE:
            x_img = np.zeros((len(indexes), IMG_HIGH, IMG_WIDE, 3), dtype='float32')
        
        y = np.zeros((len(indexes), len(TARGETS)),dtype='float32')
        sample_weights = np.zeros((len(indexes), 1),dtype='float32')

        targets_batch = list()
            
        for j, i in enumerate(indexes):
            row = self.dataframe.iloc[i]
            sign_id = row.sign_id
            if self.mode != 'test':
                sample_weight = sum(row[TARGETS_RAW].values)/20
                # sample_weight = min(sum(row[TARGETS_RAW].values)/20, 1.0)
                # sample_weight = (np.float64(sum(row[TARGETS_RAW].values) >= 10) + 0.2) / (1 + 0.2)
                
                targets_batch.append(row.expert_consensus)
            
            
            if self.mode == 'test':
                r_spe = 0
                r_eeg = 0
                r_stft = 0
            else:
                rows = df.loc[(df.eeg_id == row.eeg_id) * (df.seizure_vote == row.seizure_vote_raw) * (df.lpd_vote == row.lpd_vote_raw) * (df.gpd_vote == row.gpd_vote_raw) * (df.lrda_vote == row.lrda_vote_raw) * (df.grda_vote == row.grda_vote_raw), :].reset_index(drop=True)
                if self.mode == 'train':
                    rows = rows.iloc[np.random.permutation(len(rows))].reset_index(drop=True)
                    row = rows.loc[0, :]
                elif self.mode == 'valid':
                    row = rows.sort_values(by='eeg_sub_id').reset_index(drop=True).iloc[len(rows)//2]
                r_spe = round( row.spectrogram_label_offset_seconds / 2 )
                r_eeg = row.eeg_label_offset_seconds
                if self.mode == 'train':
                    r_eeg = r_eeg + np.random.random() * 10 - 5
                    r_eeg = max(0, r_eeg)
                    r_eeg = min(r_eeg, self.eegs[row.eeg_id].shape[1] / RSFREQ - 50)
            
            
            if 'spe' in DATATYPE:
                spe = list() # LL RL LP RP
                for k in range(4):
                    spe.append(np.reshape(self.specs[row.spectrogram_id][r_spe:(r_spe+300), k*100:(k+1)*100].T, (1, 100, 300)))
                spe = np.concatenate(spe, axis=0)
 

            if 'eeg' in DATATYPE:
                eeg = self.eegs[row.eeg_id][:, round(r_eeg * RSFREQ):round((r_eeg + 50) * RSFREQ)]
                eeg = np.concatenate((eeg[0:round(EEG_CHANNEL_USED/2), :], eeg[-round(EEG_CHANNEL_USED/2):, :]), axis=0)

            if 'stft' in DATATYPE:
                # eeg2 = torch.from_numpy(eeg.copy())
                # n_fft, win_length, hop_length = 500, 128, 50
                # ss = torchaudio.transforms.Spectrogram(n_fft=n_fft, win_length=win_length, hop_length=hop_length, power=1)(eeg2)
                # ss = ss.numpy()
                
                stft_t = self.stfts[-row.eeg_id]
                r_stft = (np.where(stft_t >= (r_eeg - min(stft_t))))[0][0]
                r_stft2 = (np.where(stft_t <= (50 + r_eeg - min(stft_t))))[0][-1]
                stft = self.stfts[row.eeg_id][:, :, r_stft:r_stft2]
                
                if stft.shape[2] < STFT_WIDE:
                    stft = np.concatenate((stft, stft[:, :, ::-1]), 2)
                stft = stft[:, :, :STFT_WIDE]


            if 'img' in DATATYPE:
                img = self.imgs[sign_id]
            
            
            # process
            if 'spe' in DATATYPE:
                spe[np.isnan(spe)] = 0
                exp_min, exp_max = -4, 6
                spe = np.clip(spe, a_min=np.exp(exp_min), a_max=np.exp(exp_max))
                spe = np.log(spe)
                
                if (spe.shape[1] != SPE_HIGH) or (spe.shape[2] != SPE_WIDE):
                    spe2 = np.zeros((spe.shape[0], SPE_HIGH, SPE_WIDE), dtype=np.float32)
                    for k in range(4):
                        scaled_arr = zoom(spe[k], (SPE_HIGH/spe.shape[1], SPE_WIDE/spe.shape[2]), order=1)
                        spe2[k, :, :] = scaled_arr
                    spe = spe2.copy()
                    
                    # spe = list() # LL RL LP RP
                    # for k in range(4):
                    #     spe.append(np.reshape(spectrograms[list(spectrograms.keys())[0]][0:(0+300), k*100:(k+1)*100].T, (1, 100, 300)))
                    # spe = np.concatenate(spe, axis=0)
                
                # spe = spe[:, :, round((spe.shape[2] - SPE_WIDE) / 2):-round((spe.shape[2] - SPE_WIDE) / 2)]
                
                if self.mode=='train':
                    spe2 = spe.copy()
                    if np.random.rand() > 0.5:
                        spe[0] = spe2[2]
                        spe[2] = spe2[0]
                    if np.random.rand() > 0.5:
                        spe[1] = spe2[3]
                        spe[3] = spe2[1]
                    if np.random.rand() > 0.5:
                        spe = spe[::-1, :, :]
                    
                    
                    if np.random.rand() > 0.5:
                        for ii in range(spe.shape[0]):
                            m1 = round(np.random.rand() * spe.shape[2] / 2)
                            m2 = round(np.random.rand() * spe.shape[2] / 2)
                            if np.random.rand() > 0.5:
                                m1 = spe.shape[2] - m1
                                m2 = spe.shape[2] - m2
                            m_min = min(m1, m2)
                            m_max = min(max(m1, m2), m_min + round(spe.shape[2] * 0.05))
                            spe[ii, :, m_min:m_max] = 0

                
                # spe = (spe - np.min(spe)) / (np.max(spe) - np.min(spe) + 1e-6) * 255
                spe = (spe - exp_min) / (exp_max - exp_min) * 255
                spe = np.clip(spe, a_min=0, a_max=255)
                
                x_spe[j] = spe

            if 'eeg' in DATATYPE:
                eeg = eeg[:, round((EEG_LENGTH - EEG_LENGTH_USED) * RSFREQ / 2):round((EEG_LENGTH + EEG_LENGTH_USED) * RSFREQ / 2)]
                # eeg_save = np.zeros((x_eeg.shape[1], x_eeg.shape[2]), dtype=np.float32)
                
                
                
                # if (np.max(eeg) < 255) and (np.min(eeg) > -255):
                #     eeg = eeg / (max(np.max(eeg), -np.min(eeg)) + 1e-6) * 255
                
                
                
                if self.mode=='train':
                    if np.random.rand() > 0.5:
                        mask = round(np.random.rand() * eeg.shape[1])
                        eeg[:, mask:round(mask + np.random.rand() * eeg.shape[1] * 0.02)] = 0
                        
                    if np.random.rand() > 0.5:
                        mask = round(np.random.rand() * eeg.shape[1])
                        eeg[:, mask:round(mask + np.random.rand() * eeg.shape[1] * 0.02)] = 0
                        
                    if np.random.rand() > 0.5:
                        mask = round(np.random.rand() * eeg.shape[1])
                        eeg[:, mask:round(mask + np.random.rand() * eeg.shape[1] * 0.02)] = 0
                    
                    
                    
                    if np.random.rand() > 0.5:
                        eeg[np.random.permutation(eeg.shape[0])[0], :] = 0
            
                    if np.random.rand() > 0.5:
                        eeg[np.random.permutation(eeg.shape[0])[0], :] = 0
                    
                    
                    # for aug in range(eeg.shape[0]):
                    #     if np.random.rand() < 1/3:
                    #         eeg[aug, :] = 0
                    
                    
                    


                    eeg[0:round(EEG_CHANNEL_USED/2), :] = eeg[0:round(EEG_CHANNEL_USED/2), :][np.random.permutation(8), :]
                    eeg[-round(EEG_CHANNEL_USED/2):, :] = eeg[-round(EEG_CHANNEL_USED/2):, :][np.random.permutation(8), :]
                    
                    eeg2 = eeg.copy()
                    eeg[4:8, :] = eeg2[12:16, :]
                    eeg[8:12, :] = eeg2[4:8, :]
                    eeg[12:16, :] = eeg2[8:12, :]
                    
                    if np.random.rand() > 0.5:
                        eeg = eeg[::-1, :]
                    
                    
                    
                    
                    
                    if np.random.rand() > 0.5:
                        eeg = -eeg
                    
                    if np.random.rand() > 0.5:
                        eeg = eeg[:, ::-1]
                    
                    
                    
                    
                    # # 添加高斯噪声
                    # eeg = eeg + np.random.normal(0, 1, eeg.shape)

                    
                else:
                    eeg2 = eeg.copy()
                    eeg[4:8, :] = eeg2[12:16, :]
                    eeg[8:12, :] = eeg2[4:8, :]
                    eeg[12:16, :] = eeg2[8:12, :]

                
                # eeg = np.clip(eeg, a_min=-255, a_max=255)
                # eeg = eeg + 255
                # eeg = eeg / 2
                
                eeg = np.clip(eeg, a_min=-1024, a_max=1024)
                eeg = eeg + 1024
                eeg = eeg / 2048 * 255

                x_eeg[j] = eeg
                
                # eeg10 = eeg[:, round((EEG_LENGTH_USED - 10) / 2 * RSFREQ):round((EEG_LENGTH_USED + 10) / 2 * RSFREQ)]
                # eeg10 = np.abs(np.corrcoef(eeg10 + 1e-6, rowvar=True))
                # eeg10 = np.where(np.isnan(eeg10), 0, eeg10)
                # # eeg10 = eeg10 * 255
                # # print(eeg10)
                # x_eeg10[j] = eeg10
                
                
                
                
            if 'stft' in DATATYPE:
                # exp_min, exp_max = -6, 6
                # stft = np.clip(stft, a_min=np.exp(exp_min), a_max=np.exp(exp_max))
                # stft = np.log(stft)
                
                exp_min, exp_max = 0, 8
                stft = np.clip(stft, a_min=0, a_max=np.exp(exp_max))
                stft = np.log1p(stft)
                
                # exp_min, exp_max = 0, 10000
                # stft = np.clip(stft, a_min=exp_min, a_max=exp_max)
                
                if self.mode=='train':
                    stft[0:round(stft.shape[0]/2), :, :] = stft[0:round(stft.shape[0]/2), :, :][np.random.permutation(stft.shape[0]//2), :, :]
                    stft[-round(stft.shape[0]/2):, :, :] = stft[-round(stft.shape[0]/2):, :, :][np.random.permutation(stft.shape[0]//2), :, :]
                    
                    stft2 = stft.copy()
                    stft[0 * stft.shape[0] // 4 : (1 * stft.shape[0] // 4)] = stft2[0 * stft.shape[0] // 4 : (1 * stft.shape[0] // 4)]
                    stft[1 * stft.shape[0] // 4 : (2 * stft.shape[0] // 4)] = stft2[3 * stft.shape[0] // 4 : (4 * stft.shape[0] // 4)]
                    stft[2 * stft.shape[0] // 4 : (3 * stft.shape[0] // 4)] = stft2[1 * stft.shape[0] // 4 : (2 * stft.shape[0] // 4)]
                    stft[3 * stft.shape[0] // 4 : (4 * stft.shape[0] // 4)] = stft2[2 * stft.shape[0] // 4 : (3 * stft.shape[0] // 4)]
                    
                    if np.random.rand() > 0.5:
                        stft = stft[::-1, :, :]
                    
                    # if np.random.rand() < 3/4:
                    #     for ii in range(stft.shape[0]):
                    #         if np.random.rand() < 1/4:
                    #             m1 = round(np.random.rand() * stft.shape[2] / 2)
                    #             m2 = round(np.random.rand() * stft.shape[2] / 2)
                    #             if np.random.rand() > 0.5:
                    #                 m1 = stft.shape[2] - m1
                    #                 m2 = stft.shape[2] - m2
                    #             m_min = min(m1, m2)
                    #             m_max = min(max(m1, m2), m_min + round(stft.shape[2] * 0.05))
                    #             stft[ii, :, m_min:m_max] = 0
                                
                    #         if np.random.rand() < 1/4:
                    #             m1 = round(np.random.rand() * stft.shape[1])
                    #             m2 = round(np.random.rand() * stft.shape[1])
                    #             m_min = min(m1, m2)
                    #             m_max = min(max(m1, m2), m_min + round(stft.shape[1] * 0.05))
                    #             stft[ii, m_min:m_max, :] = 0
                
                else:
                    stft2 = stft.copy()
                    stft[0 * stft.shape[0] // 4 : (1 * stft.shape[0] // 4)] = stft2[0 * stft.shape[0] // 4 : (1 * stft.shape[0] // 4)]
                    stft[1 * stft.shape[0] // 4 : (2 * stft.shape[0] // 4)] = stft2[3 * stft.shape[0] // 4 : (4 * stft.shape[0] // 4)]
                    stft[2 * stft.shape[0] // 4 : (3 * stft.shape[0] // 4)] = stft2[1 * stft.shape[0] // 4 : (2 * stft.shape[0] // 4)]
                    stft[3 * stft.shape[0] // 4 : (4 * stft.shape[0] // 4)] = stft2[2 * stft.shape[0] // 4 : (3 * stft.shape[0] // 4)]
                
                # stft = (stft - np.mean(stft)) / (np.std(stft) + 1e-6)
                stft = (stft - exp_min) / (exp_max - exp_min) * 255
                stft = np.clip(stft, a_min=0, a_max=255)
                
                if 'stft' in DATATYPE:
                    if j==0:
                        x_stft = np.zeros((len(indexes), stft.shape[0], stft.shape[1], stft.shape[2]), dtype='float32')
                        x_stft[j] = stft
                    else:
                        x_stft[j] = stft
            
            
            
            
            if 'img' in DATATYPE:
                img_save = np.zeros((IMG_HIGH, IMG_WIDE), dtype=np.float32)
                
                if self.mode=='train':
                    img[0:8, :, :] = img[0:8, :, :][np.random.permutation(8), :, :]
                    img[10:18, :, :] = img[10:18, :, :][np.random.permutation(8), :, :]
                    if np.random.rand() > 0.5:
                        img = img[::-1, :, :]

                for ii in range(img.shape[0]):
                    axis_temp = img_save.shape[1] / img.shape[0] / 2  * (2 * ii + 1)
                    start_temp = round(max(axis_temp - img_save.shape[1] / img.shape[0], 0))
                    end_temp = round(min(img_save.shape[0], axis_temp + img_save.shape[1] / img.shape[0]))
                    temp_temp = round(img.shape[1] / 2 - (axis_temp - start_temp))
                    img_save[start_temp:end_temp, :] = img_save[start_temp:end_temp, :] + img[ii, temp_temp:round(temp_temp + end_temp - start_temp), :]
                img_save = np.clip(img_save, a_min=0, a_max=1)
                
                img = np.reshape(img_save, (img_save.shape[0], img_save.shape[1], 1))
                img = np.concatenate((img, img, img), -1)
                
                img = (img - np.mean(img)) / (np.std(img) + 1e-6)
                
                x_img[j] = img
            
            if self.mode!='test':
                y[j] = row[TARGETS].values / sum(row[TARGETS].values)
                
                if self.sample_weights:
                    sample_weights[j] = sample_weight
                else:
                    sample_weights[j] = 1
        
        
        # if (self.mode != 'test') and (self.sample_weights):
        #     for j in range(len(sample_weights)):
        #         sample_weights[j] = sample_weights[j] / sum(np.array(targets_batch) == targets_batch[j]) / len(np.unique(targets_batch)) * len(targets_batch)
            
        
        x = {}
        if 'spe' in DATATYPE:
            x['spe'] = x_spe
        if 'eeg' in DATATYPE:
            x['eeg'] = x_eeg
            # x['eeg10'] = x_eeg10
        if 'stft' in DATATYPE:
            x['stft'] = x_stft
        if 'img' in DATATYPE:
            x['img'] = x_img
        
        return x, y, sample_weights

    







#%% BUILD MODEL
class CosineAnnealingLRScheduler(optimizers.schedules.LearningRateSchedule):
    def __init__(self, total_step, lr_max, lr_min=0, warmth_rate=0):
        super(CosineAnnealingLRScheduler, self).__init__()
        self.total_step = total_step

        if warmth_rate == 0:
            self.warm_step = 1
        else:
            self.warm_step = int(warmth_rate)

        self.lr_max = lr_max
        self.lr_min = lr_min
        
        self.begin = 1
        
    def __call__(self, step):
        if step == self.total_step:
            self.begin = 0
            # self.lr_max = self.lr_max * 0.5
            # self.lr_min = self.lr_min * 0.1
            self.lr_max = self.lr_max * 0.5
            self.lr_min = self.lr_min * 0.1
            
        step = step % self.total_step
        step = step + 1
        
        # if step < self.warm_step:
        #     lr = self.lr_max / self.warm_step * step
        # else:
        #     if self.total_step == 1:
        #         lr = self.lr_max
        #     else:
        #         lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1.0 + tf.cos((step - self.warm_step) / (self.total_step-self.warm_step) * np.pi))
                
        if (self.begin==1) and (step < self.warm_step):
            lr = self.lr_max / self.warm_step * step
        else:
            if self.begin==1:
                if self.total_step == 1:
                    lr = self.lr_max
                else:
                    lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1.0 + tf.cos((step - self.warm_step) / (self.total_step-self.warm_step) * np.pi))
            else:
                lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1.0 + tf.cos(step / 10 * np.pi))
        
        return np.float32(lr)


class IniToOne(tf.keras.initializers.Initializer):
    def __init__(self):
        super(IniToOne, self).__init__()

    def __call__(self, shape, dtype=None):
       assert len(shape) == 3
       filter_length, input_channel, filter_count = shape
       
       kernel = np.zeros(shape, dtype=np.float32)
       for i in range(filter_count):
           kernel[i%filter_length, 0, i] = 1.0
       kernel = tf.convert_to_tensor(kernel, dtype=dtype)
       return kernel

    def get_config(self):
        return {}


class SumToOne(tf.keras.constraints.Constraint):
    def __init__(self):
        super(SumToOne, self).__init__()

    def __call__(self, w):
        w = tf.abs(w)
        w_normed = w / tf.reduce_sum(w, axis=[0, 1], keepdims=True)
        return w_normed

    def get_config(self):
        return {}


class IniToOneAtten(tf.keras.initializers.Initializer):
    def __init__(self):
        super(IniToOneAtten, self).__init__()

    def __call__(self, shape, dtype=None):
       assert len(shape) == 3
       filter_length, input_channel, filter_count = shape
       
       kernel = np.zeros(shape, dtype=np.float32)
       kernel[(filter_length-1)//2:(filter_length)//2+1, :, :] = 1/((filter_length)//2+1 - (filter_length-1)//2)
       kernel = tf.convert_to_tensor(kernel, dtype=dtype)
       return kernel

    def get_config(self):
        return {}

class SumToOneAtten(tf.keras.constraints.Constraint):
    def __init__(self):
        super(SumToOneAtten, self).__init__()

    def __call__(self, w):
        w = tf.abs(w)
        w_normed = w / tf.reduce_sum(w, axis=[0, 1], keepdims=True)
        return w_normed

    def get_config(self):
        return {}



class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, feat_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential(
            [tf.keras.layers.Dense(ff_dim, activation="gelu"),
             tf.keras.layers.Dense(feat_dim),]
        )
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output, weights = self.att(inputs, inputs, return_attention_scores=True)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output), weights



class ClassToken(tf.keras.layers.Layer):
    """Append a class token to an input layer."""
    def build(self, input_shape):
        cls_init = tf.zeros_initializer()
        self.hidden_size = input_shape[-1]
        self.cls = tf.Variable(
            name="cls",
            initial_value=cls_init(shape=(1, 1, self.hidden_size), dtype="float32"),
            trainable=True,
        )
    def call(self, inputs):
        batch_size = tf.shape(inputs)[0]
        cls_broadcasted = tf.cast(
            tf.broadcast_to(self.cls, [batch_size, 1, self.hidden_size]),
            dtype=inputs.dtype,
        )
        return tf.concat([cls_broadcasted, inputs], 1)



def build_model():
    inp = list()
    y = 0
    if 'spe' in DATATYPE:
        inp_spe = tf.keras.Input(shape=(4, SPE_HIGH, SPE_WIDE), name='spe')
        x_spe = tf.keras.layers.Reshape((inp_spe.shape[1], inp_spe.shape[2], inp_spe.shape[3], 1))(inp_spe)
        x_spe = tf.keras.layers.Concatenate(axis=-1)([x_spe, x_spe, x_spe])
        
        x_spe = tf.keras.layers.Concatenate(axis=1)([x_spe[:, 0, :, :, :], x_spe[:, 1, :, :, :],
                                                     x_spe[:, 2, :, :, :], x_spe[:, 3, :, :, :]])
        
        base_model_spe = tf.keras.applications.EfficientNetV2B0(include_top=False, weights=None,
                                                                include_preprocessing=True)
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_spe.load_weights(f'./input/pre-trained-weights/{base_model_spe.name}_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_spe.load_weights(f'/kaggle/input/pre-trained-weights/{base_model_spe.name}_notop.h5')
        base_model_spe.name = 'spe_extractor'
        
        x_spe = base_model_spe(x_spe)
        
        # x_spe = x_spe[:, :, (x_spe.shape[2]-1)//2:(x_spe.shape[2])//2+1, :]
        
        x_spe = tf.keras.layers.GlobalAveragePooling2D()(x_spe)
        x_spe = tf.keras.layers.Dropout(0.5)(x_spe)
        
        inp.append(inp_spe)
        
        # y_spe = tf.keras.layers.Dense(len(TARGETS), activation='softmax', dtype='float32')(x_spe)
        y = x_spe * 1
        

    
    if 'eeg' in DATATYPE:
        inp_eeg = tf.keras.Input(shape=(EEG_CHANNEL_USED * EEG_MULTIPLY, round(EEG_LENGTH_USED * RSFREQ / EEG_MULTIPLY)), name='eeg')
        x_eeg_raw = tf.keras.layers.Reshape((inp_eeg.shape[1], inp_eeg.shape[2], 1))(inp_eeg)

        strides = 10
        if PLATFORM == 'local':
            eeg_embed = tf.keras.layers.Conv1D(filters=strides*3, kernel_size=strides, strides=strides,
                                               padding='same', use_bias=False, activation=None,
                                               kernel_initializer = IniToOne(),
                                               kernel_constraint = SumToOne(),
                                               input_shape=(None, 1)
                                               )
        else:
            eeg_embed = tf.keras.layers.Conv1D(filters=strides*3, kernel_size=strides, strides=strides,
                                                padding='same', use_bias=False, activation=None)
        
        x_eeg = tf.keras.layers.TimeDistributed(eeg_embed)(x_eeg_raw)
        
        x_eeg = tf.keras.layers.Concatenate(axis=-1)([tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1, 1))(x_eeg[:, :, :, 0*strides:1*strides]),
                                                      tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1, 1))(x_eeg[:, :, :, 1*strides:2*strides]),
                                                      tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1, 1))(x_eeg[:, :, :, 2*strides:3*strides])
                                                      ])
        x_eeg = tf.keras.layers.Permute([4, 2, 1, 3])(x_eeg)
        x_eeg = tf.keras.layers.Reshape((x_eeg.shape[1], x_eeg.shape[2], -1))(x_eeg)
        x_eeg = tf.keras.layers.Permute((3, 2, 1))(x_eeg)

        base_model_eeg = tf.keras.applications.EfficientNetV2B3(include_top=False, weights=None,
                                                                include_preprocessing=True)
        
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_eeg.load_weights(f'./input/pre-trained-weights/{base_model_eeg.name}_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_eeg.load_weights(f'/kaggle/input/pre-trained-weights/{base_model_eeg.name}_notop.h5')
        base_model_eeg.name = 'eeg_extractor'

        x_eeg = base_model_eeg(x_eeg)
        
        x_eeg = x_eeg[:, :, (x_eeg.shape[2]-1)//2:(x_eeg.shape[2])//2+1, :]
        
        x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)
        x_eeg = tf.keras.layers.Dropout(0.5)(x_eeg)

        inp.append(inp_eeg)
        
        
        
        
        
        
        # inp_eeg10 = tf.keras.Input(shape=(EEG_CHANNEL_USED, EEG_CHANNEL_USED), name='eeg10')
        # # x_eeg_raw10 = tf.keras.layers.Reshape((inp_eeg10.shape[1], inp_eeg10.shape[2], 1))(inp_eeg10)
        # x_eeg_raw10 = tf.keras.layers.Flatten()(inp_eeg10)
        # # strides = 2
        # # if PLATFORM == 'local':
        # #     eeg_embed = tf.keras.layers.Conv1D(filters=strides*3, kernel_size=strides, strides=strides,
        # #                                        padding='same', use_bias=False, activation=None,
        # #                                        kernel_initializer = IniToOne(),
        # #                                        kernel_constraint = SumToOne(),
        # #                                        input_shape=(None, 1)
        # #                                        )
        # # else:
        # #     eeg_embed = tf.keras.layers.Conv1D(filters=strides*3, kernel_size=strides, strides=strides,
        # #                                         padding='same', use_bias=False, activation=None)
        
        # # x_eeg10 = tf.keras.layers.TimeDistributed(eeg_embed)(x_eeg_raw10)
        
        # # x_eeg10 = tf.keras.layers.Concatenate(axis=-1)([tf.keras.layers.Reshape((x_eeg10.shape[1], x_eeg10.shape[2], -1, 1))(x_eeg10[:, :, :, 0*strides:1*strides]),
        # #                                               tf.keras.layers.Reshape((x_eeg10.shape[1], x_eeg10.shape[2], -1, 1))(x_eeg10[:, :, :, 1*strides:2*strides]),
        # #                                               tf.keras.layers.Reshape((x_eeg10.shape[1], x_eeg10.shape[2], -1, 1))(x_eeg10[:, :, :, 2*strides:3*strides])
        # #                                               ])
        # # x_eeg10 = tf.keras.layers.Permute([4, 2, 1, 3])(x_eeg10)
        # # x_eeg10 = tf.keras.layers.Reshape((x_eeg10.shape[1], x_eeg10.shape[2], -1))(x_eeg10)
        # # x_eeg10 = tf.keras.layers.Permute((3, 2, 1))(x_eeg10)
        # # x_eeg10 = tf.keras.layers.Concatenate(axis=-1)([x_eeg_raw10, x_eeg_raw10, x_eeg_raw10])
        

        # # base_model_eeg = tf.keras.applications.EfficientNetV2B0(include_top=False, weights=None,
        # #                                                         include_preprocessing=True)
        
        # # if NEEDTRAIN:
        # #     if PLATFORM == 'local':
        # #         base_model_eeg.load_weights(f'./input/pre-trained-weights/{base_model_eeg.name}_notop.h5')
        # #     if PLATFORM == 'kaggle':
        # #         base_model_eeg.load_weights(f'/kaggle/input/pre-trained-weights/{base_model_eeg.name}_notop.h5')
        # # base_model_eeg.name = 'eeg_extractor10'

        # # x_eeg10 = base_model_eeg(x_eeg10)
        
        # # x_eeg10 = tf.keras.layers.GlobalAveragePooling2D()(x_eeg10)
        # # x_eeg10 = tf.keras.layers.Dropout(0.5)(x_eeg10)

        # inp.append(inp_eeg10)
        
        
        
        # # x_eeg = x_eeg + x_eeg10
        # x_eeg = tf.keras.layers.Concatenate(axis=1)([x_eeg, x_eeg_raw10])
        # # x_eeg = x_eeg10
        
        
        
        y_eeg = tf.keras.layers.Dense(len(TARGETS), activation='softmax', dtype='float32')(x_eeg)
        
        
        
 
    
    if 'stft' in DATATYPE:
        inp_stft = tf.keras.Input(shape=(16, STFT_HIGH, STFT_WIDE), name='stft')
        x_stft = tf.keras.layers.Reshape((inp_stft.shape[1], inp_stft.shape[2], inp_stft.shape[3], 1))(inp_stft)
        x_stft = tf.keras.layers.Concatenate(axis=-1)([x_stft, x_stft, x_stft])
        
        x_stft = tf.keras.layers.Concatenate(axis=1)([x_stft[:, 0, :, :, :], x_stft[:, 1, :, :, :], x_stft[:, 2, :, :, :], x_stft[:, 3, :, :, :],
                                                      x_stft[:, 4, :, :, :], x_stft[:, 5, :, :, :], x_stft[:, 6, :, :, :], x_stft[:, 7, :, :, :],
                                                      x_stft[:, 8, :, :, :], x_stft[:, 9, :, :, :], x_stft[:, 10, :, :, :], x_stft[:, 11, :, :, :],
                                                      x_stft[:, 12, :, :, :], x_stft[:, 13, :, :, :], x_stft[:, 14, :, :, :], x_stft[:, 15, :, :, :]])
        
        base_model_stft = tf.keras.applications.EfficientNetV2B0(include_top=False, weights=None,
                                                                 include_preprocessing=True)
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_stft.load_weights(f'./input/pre-trained-weights/{base_model_stft.name}_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_stft.load_weights(f'/kaggle/input/pre-trained-weights/{base_model_stft.name}_notop.h5')
        base_model_stft.name = 'stft_extractor'
        
        x_stft = base_model_stft(x_stft)
        
        # x_stft = x_stft[:, :, (x_stft.shape[2]-1)//2:(x_stft.shape[2])//2+1, :]

        x_stft = tf.keras.layers.GlobalAveragePooling2D()(x_stft)

        x_stft = tf.keras.layers.Dropout(0.5)(x_stft)
        
        inp.append(inp_stft)
        # y_stft = tf.keras.layers.Dense(len(TARGETS), activation='softmax', dtype='float32')(x_stft)
        
        if y == 0:
            y = x_stft * 1
        else:
            y = tf.keras.layers.Concatenate(axis=1)([y, x_stft])
        
        
        
    
    if 'img' in DATATYPE:
        inp_img = tf.keras.Input(shape=(IMG_HIGH, IMG_WIDE, 3), name='img')
        
        
        base_model_img = tf.keras.applications.EfficientNetB0(include_top=False, weights=None,
                                                              input_tensor=inp_img)
        if NEEDTRAIN:
            if PLATFORM == 'local':
                base_model_img.load_weights(f'./input/pre-trained-weights/{base_model_img.name}_notop.h5')
            if PLATFORM == 'kaggle':
                base_model_img.load_weights(f'/kaggle/input/pre-trained-weights/{base_model_img.name}_notop.h5')
        base_model_img.name = 'img_extractor'
        x_img = base_model_img.output
        
        
        
        x_img = tf.keras.layers.GlobalAveragePooling2D()(x_img)

        # x_img = tf.keras.layers.Dropout(0.5)(x_img)
        # x_img = tf.nn.l2_normalize(x_img, -1)
        
        inp.append(inp_img)
        # if ('spe' in DATATYPE) or ('eeg' in DATATYPE) or ('stft' in DATATYPE):
        #     y = tf.keras.layers.Concatenate(axis=1)([y, x_img])
        # else:
        #     y = x_img
        # y_img = tf.keras.layers.Dense(len(TARGETS), activation='softmax', dtype='float32')(x_img)
        
        if y == 0:
            y = x_img * 1
        else:
            y = tf.keras.layers.Concatenate(axis=1)([y, x_img])

    # y = tf.keras.layers.Dense(len(TARGETS), activation='softmax', dtype='float32')(y)
    
    y = y_eeg * 1

    model = tf.keras.Model(inputs=inp, outputs=y)
        
    return model







#%% train_fold
def train_fold(i, stage, train_index, valid_index, df_train_stage1, df_valid_stage1, df_train_stage2, df_valid_stage2, 
               build_model, BATCHSIZE, EPOCHS, LEARN_RATE, TARGETS, TARGETS_RAW):
    
    # 输出日志信息
    print('#'*25)
    print(f'### Fold {i+1}')
    
    # 构建模型
    model = build_model()
    loss = tf.keras.losses.KLDivergence()
    
    # # 定义带有梯度裁剪的优化器
    # class ClipOptimizer(tf.keras.optimizers.AdamW):
    #     def __init__(self, learning_rate, clip_norm):
    #         super().__init__(learning_rate=learning_rate)
    #         self.clip_norm = clip_norm

    #     def get_gradients(self, loss, params):
    #         grads = super().get_gradients(loss, params)
    #         clipped_grads, _ = tf.clip_by_global_norm(grads, self.clip_norm)
    #         return clipped_grads

    # clip_norm_value = 1.0  # 调整这个值以控制梯度裁剪的强度
    
    
    # 构建数据生成器
    if stage == 1:
        train_gen_stage = DataGenerator(df_train_stage1, shuffle=True, sample_weights=True, batch_size=BATCHSIZE, specs=spectrograms, eegs=eegs, stfts=stfts, imgs=imgs)
        valid_gen_stage = DataGenerator(df_valid_stage1, shuffle=False, sample_weights=True, batch_size=BATCHSIZE*2, mode='valid', specs=spectrograms, eegs=eegs, stfts=stfts, imgs=imgs)
        opt = tf.keras.optimizers.AdamW(learning_rate=LEARN_RATE)
        # opt = ClipOptimizer(learning_rate=LEARN_RATE, clip_norm=clip_norm_value)
        # 设置回调函数
        callbacks_stage = [
            tf.keras.callbacks.LearningRateScheduler(CosineAnnealingLRScheduler(EPOCHS, LEARN_RATE, LEARN_RATE * 0.1 * 0.1, 5)),
            tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join('models', f'fold{i}_stage1.weights.h5'),
                                               monitor='val_loss', mode='min',
                                               save_weights_only=True, save_best_only=True)
        ]
    elif stage == 2:
        train_gen_stage = DataGenerator(df_train_stage2, shuffle=True, sample_weights=False, batch_size=BATCHSIZE, specs=spectrograms, eegs=eegs, stfts=stfts, imgs=imgs)
        valid_gen_stage = DataGenerator(df_valid_stage2, shuffle=False, sample_weights=False, batch_size=BATCHSIZE*2, mode='valid', specs=spectrograms, eegs=eegs, stfts=stfts, imgs=imgs)
        opt = tf.keras.optimizers.AdamW(learning_rate=LEARN_RATE * 0.1)
        # opt = ClipOptimizer(learning_rate=LEARN_RATE * 0.1, clip_norm=clip_norm_value)
        model.load_weights(os.path.join('models', f'fold{i}_stage1.weights.h5'))
        # 设置回调函数
        callbacks_stage = [
            tf.keras.callbacks.LearningRateScheduler(CosineAnnealingLRScheduler(max(round(EPOCHS/3), 1), LEARN_RATE * 0.1, LEARN_RATE * 0.1 * 0.1, 0)),
            tf.keras.callbacks.ModelCheckpoint(filepath=os.path.join('models', f'fold{i}_stage2.weights.h5'),
                                               monitor='val_loss', mode='min',
                                               save_weights_only=True, save_best_only=True)
        ]

    model.compile(loss=loss, optimizer=opt)
    
    # 冻结参数
    # for layer_i, layer in enumerate(model.layers):
    #     if layer.name[:16] == 'time_distributed':
    #         model.layers[layer_i].trainable = False
    
    # 开始训练
    if stage == 1:
        history = model.fit(train_gen_stage, verbose=1, validation_data=valid_gen_stage,
                            epochs=EPOCHS, callbacks=callbacks_stage)
    elif stage == 2:
        history = model.fit(train_gen_stage, verbose=1, validation_data=valid_gen_stage,
                            epochs=max(round(EPOCHS/3), 1), callbacks=callbacks_stage)

    # 加载最优权重
    model.load_weights(os.path.join('models', f'fold{i}_stage{stage}.weights.h5'))
 
    # 绘制损失函数图像
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(loss) + 1)
    plt.figure()
    plt.plot(epochs, loss, 'bo', label='loss')
    plt.plot(epochs, val_loss, 'b', label='val_loss')
    plt.title(f'loss: {round(min(loss), 4)}, val loss: {round(min(val_loss), 4)}', fontsize=12)
    plt.legend()
    plt.savefig(os.path.join('models', f'fold{i}_stage{stage}.svg'))
    plt.close()

    # 对验证集进行预测并生成混淆矩阵
    if stage == 1:
        valid_stage = df_valid_stage1[TARGETS].values
    elif stage == 2:
        valid_stage = df_valid_stage2[TARGETS].values
    predict_stage = model.predict(valid_gen_stage)
        
    # 删除数据生成器和历史记录
    del train_gen_stage, valid_gen_stage, history, model
    tf.keras.backend.clear_session()
    gc.collect()

    cm = confusion_matrix(np.argmax(valid_stage, 1), np.argmax(predict_stage, 1))
    cm = cm / np.sum(cm, 1, keepdims=True)
        
    plt.figure()
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(6)
    plt.xticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
    plt.yticks(tick_marks, [f'{TARGETS[i][:-5]}' for i in [0, 1, 2, 3, 4, 5]], fontsize=10)
    thresh = cm.max() / 2.
    for ii, jj in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if cm[ii, jj] > -0.1:
            plt.text(jj, ii, str(round(cm[ii, jj] * 1e4) * 1e-2)[:5], horizontalalignment="center", color="white" if cm[ii, jj] > thresh else "black", fontsize=10)
    plt.xlabel('Predicted label')
    plt.ylabel('True label')
    plt.tight_layout()
    plt.savefig(os.path.join('models', f'fold{i}_stage{stage}_cm.svg'))
    plt.close()

    # 清理资源
    del df_train_stage1, df_valid_stage1, df_train_stage2, df_valid_stage2
    gc.collect()







#%% TRAIN MODEL
if __name__ == '__main__':
    if NEEDTRAIN:
        if not os.path.exists('models'):
            os.makedirs('models')
        
        from sklearn.model_selection import GroupKFold
        import multiprocessing as mp
        mp.set_start_method('spawn')  # 设置进程启动方式为 spawn
        
        gkf = GroupKFold(n_splits=SPLITS)
        # train = train.loc[np.random.permutation(len(train))].reset_index(drop=True)

        # train = train.drop_duplicates(['eeg_id']).reset_index(drop=True)
        
        for i, (train_index, valid_index) in enumerate(gkf.split(train, train.expert_consensus, train.patient_id)):  
            print('#'*25)
            print(f'### Fold {i+1}')
            
            df_train_stage1 = train.iloc[train_index].reset_index(drop=True)
            df_valid_stage1 = train.iloc[valid_index].reset_index(drop=True)

            df_train_stage2 = df_train_stage1[np.sum(df_train_stage1[TARGETS_RAW].values, 1) >= 10].reset_index(drop=True)
            df_valid_stage2 = df_valid_stage1[np.sum(df_valid_stage1[TARGETS_RAW].values, 1) >= 10].reset_index(drop=True)
        
            for stage in [1, 2]:
                p = mp.Process(target=train_fold, args=(
                    i,
                    stage,
                    train_index,
                    valid_index,
                    df_train_stage1,
                    df_valid_stage1,
                    df_train_stage2,
                    df_valid_stage2,
                    build_model,
                    BATCHSIZE,
                    EPOCHS,
                    LEARN_RATE,
                    TARGETS,
                    TARGETS_RAW,
                ))
                p.start()
                p.join()
    
    #%% INFER
    else:
        preds_all = []
        models = list()
        model_template = build_model()
        # for model_i in range(SPLITS):
        for model_i in range(100):
            if os.path.exists(os.path.join(LOAD_MODELS_FROM, f'fold{model_i}_stage2.weights.h5')):
                print(f'Fold {model_i+1}')
                model = clone_model(model_template)
                model.load_weights(os.path.join(LOAD_MODELS_FROM, f'fold{model_i}_stage2.weights.h5'))
                models.append(model)
            

        test = pd.read_csv(os.path.join(LOAD_DATA_FROM, 'test.csv'))
        test['sign_id'] = test.index.values
        print('Test shape', test.shape)

        if 'spe' in DATATYPE:
            PATH_test = os.path.join(LOAD_DATA_FROM, 'test_spectrograms') + '/'
            files_test = os.listdir(PATH_test)
            print(f'There are {len(files_test)} test spectrogram parquets')
                
            for i, f in enumerate(files_test):
                if i%100==0: print(i,', ',end='')
                tmp = pd.read_parquet(f'{PATH_test}{f}')
                name = int(f.split('.')[0])
                spectrograms_test[name] = tmp.iloc[:,1:].values

        PATH_test = os.path.join(LOAD_DATA_FROM, 'test_eegs') + '/'
        if ('spe' in DATATYPE) or ('eeg' in DATATYPE) or ('stft' in DATATYPE) or ('img' in DATATYPE):
            b2, a2 = signal.butter(3, np.float32(filter_range2)*2/RSFREQ, 'bandpass')
            
            for i, eeg_id in enumerate(test.eeg_id):
                if i%100==0: print(i,', ',end='')
                eeg_default = pd.read_parquet(os.path.join(PATH_test, (str(eeg_id) + '.parquet')))
                
                eeg = list()
                for channel in BRAIN:
                    eeg_temp = (eeg_default.loc[:, channel.split('-')[0]] - eeg_default.loc[:, channel.split('-')[1]]).values
                    eeg_temp[np.isnan(eeg_temp)] = 0
                    eeg.append(np.reshape(eeg_temp, (1, -1)))
                eeg = np.concatenate(eeg, axis=0)
                
                if SFREQ != RSFREQ:
                    eeg = signal.resample_poly(eeg, RSFREQ, SFREQ, axis=1)
                
                if 'stft' in DATATYPE:
                    eeg2 = signal.filtfilt(b, a, eeg, axis=1)

                    nperseg = round(RSFREQ * 1)
                    ff, tt, ss = signal.spectrogram(eeg2, axis=1, fs=RSFREQ, nperseg=nperseg,
                                                    noverlap = round(nperseg - RSFREQ * STFT_TIME),
                                                    # noverlap=round((1-0.1) * 0.1 * RSFREQ),
                                                    nfft = RSFREQ * 5
                                                    )
                    ss[np.isnan(ss)] = 0
                    ss = ss[:, (ff>0) * (ff<=20), :]
                    
                    ss = np.concatenate((ss[0:round(EEG_CHANNEL_USED/2), :, :], ss[-round(EEG_CHANNEL_USED/2):, :, :]), axis=0)
                    
                    for k in range(4):
                        ss[k, :, :] = np.mean(ss[k*4:(k+1)*4, :, :], 0)
                    
                    ss = ss[:4, :, :]
                    
                    ss = np.array(ss, dtype=np.float32)
                    tt = np.array(tt, dtype=np.float32)


                
                if 'img' in DATATYPE:
                    eeg2 = signal.filtfilt(b2, a2, eeg, axis=1)
                
                    eeg2 = np.clip(eeg2, a_min=-1024, a_max=1024)
                    
                    
                    train_plot = test[test.eeg_id == eeg_id].reset_index(drop=True)
                    for j in range(len(train_plot)):
                        eeg_plot = eeg2[:, 0:EEG_LENGTH * RSFREQ]
                        eeg_plot = eeg_plot[:, round((EEG_LENGTH - IMG_LENGTH) / 2 * RSFREQ):round((EEG_LENGTH + IMG_LENGTH) / 2 * RSFREQ)]
                        
                        img_save = np.zeros((eeg_plot.shape[0], 36, IMG_WIDE), dtype=np.float32)
                        for ii in range(eeg_plot.shape[0]):
                            fig = plt.figure(clear=True, figsize=(3.93, 2 / 18 * 2))
                            fig.patch.set_facecolor('black')

                            plt.plot(eeg_plot[ii, :] + 100, color='red', linewidth=0.2)
                            
                            plt.xlim(-5, eeg_plot.shape[1] + 5)
                            plt.ylim(0, 200)
                            plt.axis('off')
                            
                            byte_stream = io.BytesIO()
                            plt.savefig(byte_stream, format='png', bbox_inches='tight', dpi=100)
                            byte_stream.seek(0)
                            img = Image.open(byte_stream)
                            img = np.array(img)[:, :, :1]
                            img = img / 255
                            img = np.array(img, dtype=np.float32)
                            byte_stream.truncate()
                            plt.close('all')
                            
                            if img.shape != (36, IMG_WIDE, 1):
                                img = np.concatenate((img, img, img), 2)
                                img = np.array(tf.image.resize(img, (36, IMG_WIDE)), dtype=np.float32)
                            img = img[:, :, 0]
                            
                            img_save[ii, :, :] = img
     
        
                        imgs_test[train_plot.sign_id[j]] = img_save
                
                eeg = np.clip(eeg, a_min=-1024, a_max=1024)
                eegshape = eeg.shape[1]
                eeg = np.concatenate((eeg[:, ::-1], eeg, eeg[:, ::-1]), axis=1)
                if filter_range != None:
                    eeg = signal.filtfilt(b, a, eeg, axis=1)
                eeg = eeg[:, eegshape:eegshape*2]
                
                eeg = np.array(eeg, dtype=np.float32)
                
                if 'eeg' in DATATYPE:
                    eegs_test[eeg_id] = eeg
                if 'stft' in DATATYPE:
                    stfts_test[eeg_id] = ss
                    stfts_test[-eeg_id] = tt 
                
                
                
                if ((i+1)%TEST_BATCHSIZE==0) or ((i+1)==len(test.eeg_id)):
                    preds = []
                    test_gen = DataGenerator(test.loc[max(i-TEST_BATCHSIZE+1, len(preds_all)):i, :], shuffle=False, sample_weights=False, batch_size=TEST_BATCHSIZE, mode='test', specs=spectrograms_test, eegs=eegs_test, stfts=stfts_test, imgs=imgs_test)
                    for model_i in range(len(models)):
                        pred = models[model_i].predict(test_gen, verbose=1)
                        preds.append(pred)
                    pred = np.mean(preds, axis=0)
                    del eegs_test
                    gc.collect()
                    eegs_test = {}
                    if len(preds_all) == 0:
                        preds_all = pred.copy()
                    else:
                        preds_all = np.concatenate((preds_all, pred), axis=0)

        sub = pd.DataFrame({'eeg_id': test.eeg_id.values})
        sub[TARGETS] = preds_all
        sub.to_csv('submission.csv', index=False)
        print('Submissionn shape', sub.shape)
        sub.head()
















['eeg']


E0000 00:00:1777352414.120781      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777352414.125013      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2.18.0
[]
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Test shape (9850, 4)
0 , 100 , 200 , 

TypeError: object of type 'numpy.float64' has no len()